## 📑 HAPS Assignment Mapping
 
#### 🔗 Links
- **[BRD link](https://cepalhellas.atlassian.net/wiki/x/AQDhHQ)**  
  Comprehensive documentation outlining the business requirements for this report.

##### Authentication

In [0]:
import boto3
from botocore.exceptions import NoCredentialsError
import traceback

"""
Accessing the Hive metastore requires specific considerations:

1. Hive metastore access is not possible via the instance profile if the cluster is on shared access mode.
   The cluster must be in shared access mode to interact with the Unity catalog.

2. To access both Unity catalog and AWS services, an access token from Amazon is required.

3. To retrieve the token secret key, access the secrets scope 'aws-databricks,' where the access key is stored per environment.
    https://docs.databricks.com/en/security/secrets/index.html

4. Once AWS credentials are obtained, clients for services like S3 and SSM can be created.

5. After you authenticate successfully querying the Hive metastore is also working.
"""

try:
    mode = dbutils.widgets.get("mode")
except Exception as e:
    mode = "shared" 

# Function to get AWS credentials from Databricks secrets.
def get_aws_credentials(secrets_scope, key_name):
    try:
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None
    
# Function to get AWS credentials from Databricks secrets.
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration.
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
    
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

secrets_scope = "aws-databricks"
aws_keys = get_scope_key_names(secrets_scope)

aws_key = aws_keys[0] if aws_keys else None

# Get AWS secret key from Databricks secrets.
aws_secret_key = get_aws_credentials(secrets_scope,aws_key)

if mode == "shared":
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)

    S3_client = boto3.client('s3', region_name='eu-central-1',
                                aws_access_key_id=aws_key,
                                aws_secret_access_key=aws_secret_key)                           
else:
    # Connect to AWS using boto3.
    ssm_client = boto3.client('ssm', region_name='eu-central-1')

    S3_client = boto3.client('s3', region_name='eu-central-1')                           


# Function to connect to AWS and access SSM parameter.
def connect_to_aws_and_access_ssm(ssm_key):

    if aws_secret_key is None:
        print("AWS secret key not available.")
        return

    # Access SSM parameter /cfn/a360/reports_output_path.
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )

    return response['Parameter']['Value']

# Get the Buckewt prefix from SSM to be sure.
env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix')

In [0]:
from pyspark.sql.functions import coalesce, concat_ws, lit, sha2, substring, when, col,regexp_replace

import delta_table_creator, extract, transform
from delta import DeltaTable  # type: ignore

# from pyspark.sql import SparkSession
# spark = SparkSession.builder.appName("example").getOrCreate()

from pyspark.sql import DataFrame
from pyspark.sql.functions import trim

In [0]:
import boto3
import os
 
#takes the values d for dev, qa for test and p for prod depending in which workspace the init variables script is running
# env = os.environ["S3PREFIX"]

dbutils.widgets.text("run_date", '')
run_date = dbutils.widgets.get("run_date").strip()

if not run_date:
    run_date = spark.sql(f'''
                         select MAX(BusinessDate) BusinessDate from `{env_prefix}-cdh`.derivative.dexposure 
                         union
                         select MAX(BusinessDate) BusinessDate from `p-cdh`.landed.profits_loan_details
                         ''').collect()

    if len(run_date) != 1: raise ValueError(f"Incorrect run_date!")
    run_date = run_date[0]['BusinessDate']
    run_date = run_date.strftime('%Y-%m-%d')

In [0]:
ReferenceDateSoM = spark.sql(  
f'''
select CAST(PreviousMonthLastWorkingDate AS DATE) ReferenceDateSoM
from `{env_prefix}-cdh`.reference.ref_calendar
where CalendarDate = '{run_date}'
'''
).collect()[0]['ReferenceDateSoM']   


AssignmentCutOffDate = spark.sql(
f'''
select 
CASE WHEN '{run_date}' <
(CASE WHEN DayName  = 'Saturday' 
THEN (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year('{run_date}'), '-', month('{run_date}'),'-16')))
ELSE (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year('{run_date}'), '-', month('{run_date}'),'-15')) ) END)
THEN '{run_date}'
ELSE 
(CASE WHEN DayName  = 'Saturday' 
THEN (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year('{run_date}'), '-', month('{run_date}'),'-16')))
ELSE (SELECT NextBusinessDay FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year('{run_date}'), '-', month('{run_date}'),'-15')) ) END)
	END	as AssignmentCutOffDate
 FROM `{env_prefix}-cdh`.reference.ref_calendar WHERE CalendarDate =  DATE(CONCAT(year('{run_date}'), '-', month('{run_date}'),'-14'))
'''
).collect()[0]['AssignmentCutOffDate']   


FirstAssignmentDate = spark.sql(  
f'''
SELECT CalendarDate AS FirstAssignmentDate
FROM `{env_prefix}-cdh`.reference.ref_calendar
WHERE CalendarDate BETWEEN '{ReferenceDateSoM}' AND last_day('{AssignmentCutOffDate}') AND BusinessDayOfMonth=7
'''
).collect()[0]['FirstAssignmentDate']   

###### Schema of the table

In [0]:
# Define the schema name
schema_name = f"`{env_prefix}-cdh-delivery`.repoint"

# Create the schema if it does not exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_name}")

table_name = 'HAPS_AssignmentMapping' 

schema = (
    "BusinessDate DATE"
    ",PortfolioName STRING"
    ",ExposureKey STRING"
    ",AgreementKey STRING"
    ",CDI STRING"
    ",PartyTaxRegistrationNumber STRING"
    ",AssetSegment STRING"
    ",ProductKey STRING"
    ",SoMOnBalanceAmount DECIMAL(19, 2)"
    ",SoMOnBalanceAmountEUR DECIMAL(19, 2)"
    ",SoMGBV DECIMAL(19, 2)"
    ",SoMGBVEUR DECIMAL(19, 2)"
    ",SoMBucket STRING"
    ",SoMDaysPastDue INT"
    ",SoMExposureStatus STRING"
    ",SoMObjectiveStatusCode STRING"
    ",SoMObjectiveStatusDate DATE"
    ",MtDIsActiveFlag STRING"
    ",MtDCloseDate DATE"
    ",AssignmentChannel STRING"
    ",AssignmentTeam STRING"
    ",AssignmentAgent STRING"
    ",AssignmentAgentKey STRING"
    ",QueueDescriptionFinal STRING"
    ",CourseOfAction STRING"
    ",AssignmentSegmentDetailed STRING"
    ",CaseCode STRING"
    ",CollectionIntramonthAssignment INT"
    ",AccountCategory STRING"
    ",GModifiedAccount STRING"
    ",SegmentationBusinessDate DATE"
    ",OfficialSegment STRING"
    ",NewOrAlphaProduct STRING"
    ",PayerFlagEntity STRING"
    ",BorrowerTotalGBV DECIMAL(19, 2)"
    ",BorrowerTotalGBVRange STRING"
    ",HighBalanceFlag INT"
    ",InsertTimeStamp TIMESTAMP"

)


In [0]:
partition_columns = ["BusinessDate"]
delivery_table_path = ""
 
# ================== Create Delta Table ==================
#delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"  
#AWS Credentials not available thus env_prefix returns empty temporarily we manually set it to `qa`

delivery_table_path = f"`{env_prefix}-cdh-delivery`.repoint.{table_name}"

delta_table_creator.create_unity_delta_table(
    spark, delivery_table_path, partition_columns, schema
)
# if not DeltaTable.isDeltaTable(spark, delivery_table_path):
#     delta_table_creator.create_unity_delta_table(
#         spark, delivery_table_path, partition_columns, schema
#     )
  
# ================== Delete From Delta Table ==================
transform.delete_from_unity_table(
    spark,
    delivery_table_path,
    run_date
#### type: ignore
)


##### Create Delta Tables with selected fields & Temporary Views for each delta table

##### haps_portfolios

In [0]:
haps_portfolios = spark.sql(
f'''
SELECT DISTINCT A.PortfolioName
FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios  A
INNER JOIN (SELECT BusinessDate, MAX(IngestionTimeStamp) IngestionTimeStamp, PortfolioName 
            FROM `{env_prefix}-reportinputfiles`.repoint.haps_repointportfolios
            GROUP BY  BusinessDate, PortfolioName) B
ON A.BusinessDate = B.BusinessDate AND A.PortfolioName = B.PortfolioName AND A.IngestionTimeStamp = B.IngestionTimeStamp
WHERE A.Report = 'HAPS_AssignmentMapping'
AND A.IsActive = 1 
''' 
)
haps_portfolios.createOrReplaceTempView("haps_portfolios")


##### Segmentation Dates

In [0]:
SegmentationDates = spark.sql(
 
f'''
 
SELECT B.PortfolioName
      ,B.SegmentationDate
      ,A.ExposureKeyWithPrefix
      ,MAX(A._update_timestamp) AS _update_timestamp
FROM `{env_prefix}-ptools`.assignments.offer_collection  A
INNER JOIN (
             SELECT PortfolioName, MAX(BusinessDate) AS SegmentationDate
             FROM `{env_prefix}-ptools`.assignments.offer_collection
             WHERE BusinessDate <= CAST('{AssignmentCutOffDate}' AS DATE) AND ExposureKeyWithPrefix IS NOT NULL
             AND PortfolioName is NOT null
             GROUP BY PortfolioName
           ) B
ON A.PortfolioName  = B.PortfolioName AND A.BusinessDate = B.SegmentationDate    
WHERE A.ExposureKeyWithPrefix IS NOT NULL
AND A.AssignmentStatus = 'approved'
AND  A.RelationType = 'Primary Owner'
GROUP BY B.PortfolioName
        ,B.SegmentationDate
        ,A.ExposureKeyWithPrefix   
 
'''
 
)
 
SegmentationDates.createOrReplaceTempView("SegmentationDates")


##### PerimeterProfitsSOM

In [0]:
PerimeterProfitsSOM = spark.sql(
f'''
SELECT 
        E.BusinessDate
	, E.PortfolioName
      , E.ExposureKey                   
      , E.AgreementKey                  
      , E.CDI                           
      , E.PartyTaxRegistrationNumber    
      , E.AssetSegment
      , E.ProductKey
      , E.ExposureStatus
      , E.GBV
      , E.Currency
      , E.DaysPastDueWithMateriality
FROM `{env_prefix}-cdh`.derivative.dexposure as E
INNER JOIN haps_portfolios AS A
ON E.PortfolioName = A.PortfolioName
WHERE E.BusinessDate = '{ReferenceDateSoM}'
AND E.SourceSystem = 'profits'	
AND E.PartyRelationshipType = 'Primary Owner' 
AND E.IsActiveFlag = TRUE
'''
)

PerimeterProfitsSOM.createOrReplaceTempView("PerimeterProfitsSOM")

##### AccountsQualcoSoM

In [0]:
AccountsQualcoSoM = spark.sql(
f'''
SELECT 
        E.BusinessDate 
	, E.PortfolioName
      , E.PortfolioKey
      , E.ExposureKey                         
	, E.OnBalance
      , E.ObjectiveStatusCode
      , E.ObjectiveStatusdate
      , E.CaseCode
      , E.Bucket
FROM `{env_prefix}-cdh`.primary.exposure as E
INNER JOIN haps_portfolios AS A
ON E.PortfolioName = A.PortfolioName
where E.businessdate = '{ReferenceDateSoM}'
AND E.SourceSystem = 'qca'
AND E.PartyRelationshipType = 'Primary Owner'
'''
)

AccountsQualcoSoM.createOrReplaceTempView("AccountsQualcoSoM")

##### ClosedDateMtD

In [0]:
ClosedDateMtD = spark.sql(
f'''
SELECT  * 
    ,   RANK() OVER(PARTITION BY a.ExposureKey ORDER BY CASE WHEN a.IsActiveFlag = true THEN 1 ELSE 0 END DESC) Rank 
FROM (
SELECT 
        E.ExposureKey       
      , E.IsActiveFlag
      , MAX(E.BusinessDate)  AS ClosedDate                      
FROM `{env_prefix}-cdh`.derivative.dexposure as E
INNER JOIN haps_portfolios AS A
ON E.PortfolioName = A.PortfolioName
where E.businessdate <= '{run_date}' AND E.businessdate >= '{ReferenceDateSoM}'
AND E.PartyRelationshipType = 'Primary Owner' 
AND E.SourceSystem = 'profits'
AND E.IsActiveFlag = FALSE
GROUP BY E.ExposureKey , E.IsActiveFlag 

UNION ALL

SELECT 
        E.ExposureKey       
      , E.IsActiveFlag
      , E.ClosedDate                      
FROM `{env_prefix}-cdh`.derivative.dexposure as E
INNER JOIN haps_portfolios AS A
ON E.PortfolioName = A.PortfolioName
where E.businessdate = '{run_date}' 
AND E.PartyRelationshipType = 'Primary Owner' 
AND E.SourceSystem = 'profits'
AND E.IsActiveFlag = TRUE) a 

'''
)
ClosedDateMtD.createOrReplaceTempView("ClosedDateMtD")


##### ExchangeRatesSoM

In [0]:
ExchangeRatesSoM = spark.sql(
f'''
SELECT 
        E.BusinessDate  
      , E.OriginalCurrency                   
      , E.ExchangeRate                                       
FROM `{env_prefix}-cdh`.reference.exchangerates as E
where E.businessdate = '{ReferenceDateSoM}'
'''
)
ExchangeRatesSoM.createOrReplaceTempView("ExchangeRatesSoM")

##### QueuesMtD

In [0]:
QueuesMtD = spark.sql(
f'''
select 
				E.BusinessDate
			,	e.CaseCodeKey 
			, e.Queue
FROM `{env_prefix}-cdh`.primary.casecode AS E
INNER JOIN haps_portfolios AS A
ON E.PortfolioName = A.PortfolioName
WHERE E.businessdate >= '{ReferenceDateSoM}' and E.businessdate <= '{run_date}'
AND E.SourceSystem = 'qca'
AND E.CaseRelationTypeCode = 5000
'''
)

QueuesMtD.createOrReplaceTempView("QueuesMtD")

##### AssignmentMtD

In [0]:
AssignmentMtD = spark.sql(
f'''
-- HARDENING: Deduplicate assignment to one row per CaseCode + BusinessDate.
-- The source `primary.assignment` (external QCA files) can contain duplicate
-- rows for the same CaseCode on a given date, especially when a NEW portfolio
-- is migrated and its assignment data is not fully populated yet. Those
-- duplicates tie at RANK()=1 in AllQueuesChannels and are then squared by the
-- RnkRef self-join in QueuesChannels, producing duplicate ExposureKey rows in
-- the final table. Collapsing to one deterministic row per CaseCode+BusinessDate
-- here removes the amplification at its source.
SELECT 
	  BusinessDate
	, CaseCode 
    , ExposureKey
	, Channel
    , TeamCode
    , Team
    , AgentPartyKey
    , AgentFullName
FROM (
    SELECT 
	      E.BusinessDate
	    , E.CaseCode 
        , E.ExposureKey
	    , E.Channel
        , E.TeamCode
        , E.Team
        , E.AgentPartyKey
        , E.AgentFullName
        , ROW_NUMBER() OVER (
              PARTITION BY E.CaseCode, E.BusinessDate
              ORDER BY CASE WHEN E.Channel      IS NOT NULL THEN 0 ELSE 1 END
                     , CASE WHEN E.AgentPartyKey IS NOT NULL THEN 0 ELSE 1 END
                     , E.Channel
                     , E.AgentPartyKey
                     , E.ExposureKey
          ) AS rn
    FROM `{env_prefix}-cdh`.primary.assignment AS E
    INNER JOIN haps_portfolios AS A
    ON E.PortfolioName = A.PortfolioName
    WHERE E.businessdate >= '{ReferenceDateSoM}' and E.businessdate <= '{run_date}'
    AND E.SourceSystem = 'qca'
    AND E.AssignmentType = 'Channel Assignment'
)
WHERE rn = 1
'''
)

AssignmentMtD.createOrReplaceTempView("AssignmentMtD")

##### SegmentationSoM

In [0]:
SegmentationSoM = spark.sql(
f'''
SELECT distinct
	  E.BusinessDate
  , E.ExposureKey
	, E.segment_mapped_values        	AS OfficialSegment
	, E.NewOrAlphaProduct_calc      	AS NewOrAlphaProduct
	, E.BorrowerTotalGBV          		AS BorrowerTotalGBV
	, E.Borrower_total_GBV_Range  		AS BorrowerTotalGBVRange   
	, CASE WHEN HB.CDI IS NOT NULL THEN 1 ELSE 0 END HighBalanceFlag
FROM `{env_prefix}-ptools`.assignments.offer_collection AS E
INNER JOIN SegmentationDates as SD ON E.BusinessDate = SD.SegmentationDate 
AND E.PortfolioName = SD.PortfolioName
and e._update_timestamp = sd._update_timestamp
AND SD.ExposureKeyWithPrefix=E.ExposureKeyWithPrefix       
LEFT JOIN (
    SELECT DISTINCT
        E2.CDI
    FROM `{env_prefix}-ptools`.assignments.offer_collection AS E2
    INNER JOIN SegmentationDates AS SD2 
        ON E2.BusinessDate = SD2.SegmentationDate 
        AND E2.PortfolioName = SD2.PortfolioName
    WHERE E2.RelationType = 'Primary Owner'
        and E2.HighBalanceFlag_c = TRUE
) HB ON  E.CDI = HB.CDI
WHERE E.RelationType = 'Primary Owner'

'''
)

SegmentationSoM.createOrReplaceTempView("SegmentationSoM")

##### PayersFlagAlgo

In [ ]:
PayersFlagAlgo = spark.sql(
f'''
SELECT DISTINCT
    PF.CDI
  , PF.PortfolioName
  , PF.Payer_Flag_Entity AS PayerFlagEntity
FROM `p-ds`.aaas.payers_flag_algorithm PF
WHERE PF.BusinessDate = '{ReferenceDateSoM}'
'''
)

PayersFlagAlgo.createOrReplaceTempView("PayersFlagAlgo")

##### QueuesMapping

In [0]:
QueuesMapping = spark.sql(
f'''
SELECT 
	  e.BusinessDate
	, e.QueueDescription
  , e.Channel
	, e.AssignmentSegmentDetailed
  , e.CourseOfAction
FROM `{env_prefix}-reportinputfiles`.repoint.haps_queuesmapping AS E
INNER JOIN (SELECT MAX(IngestionTimeStamp) IngestionTimeStamp, BusinessDate FROM `{env_prefix}-reportinputfiles`.repoint.haps_queuesmapping GROUP BY BusinessDate) its ON E.IngestionTimeStamp = its.IngestionTimeStamp and E.BusinessDate = its.BusinessDate
WHERE E.BusinessDate = (SELECT MAX(BusinessDate) FROM `{env_prefix}-reportinputfiles`.repoint.haps_queuesmapping WHERE BusinessDate <= '{run_date}' ) 
'''
)

QueuesMapping.createOrReplaceTempView("QueuesMapping")

##### AllQueuesChannels

In [0]:
AllQueuesChannels = spark.sql(
f'''
SELECT 
	  Q.BusinessDate
	, Q.CaseCodeKey 
	, Q.Queue
  , A.ExposureKey
	, A.Channel
  , A.TeamCode
  , A.Team
  , A.AgentPartyKey
  , A.AgentFullName
	, RANK() OVER (PARTITION BY Q.CaseCodeKey
					ORDER BY    CASE WHEN Q.BusinessDate > '{AssignmentCutOffDate}' THEN 1 ELSE 2 END DESC,
                                CASE WHEN  
										(
											 lower(Q.Queue) NOT LIKE '%low_debt%'
										 AND lower(Q.Queue) NOT LIKE '%under_11_dpds%'
										 AND lower(Q.Queue) NOT LIKE '%_current%'
										 AND lower(Q.Queue) NOT LIKE 'emp_tempassign%'
										 AND lower(Q.Queue) NOT LIKE 'vlt / negb%'
										 AND NOT ( lower(Q.Queue) LIKE '%ext' AND A.Channel IS NULL)
												 )							THEN 1
																			ELSE 2
							   END ASC,
							   Q.BusinessDate DESC)						    AS RNKHistorical		
	, RANK() OVER (PARTITION BY Q.CaseCodeKey
		ORDER BY    CASE WHEN  Q.BusinessDate > '{AssignmentCutOffDate}' THEN 1 ELSE 2 END DESC,
                    Q.BusinessDate DESC)							        AS RNKRefDateHistorical	      
  	, RANK() OVER (PARTITION BY Q.CaseCodeKey
					ORDER BY    CASE WHEN  
										(
											 lower(Q.Queue) NOT LIKE '%low_debt%'
										 AND lower(Q.Queue) NOT LIKE '%under_11_dpds%'
										 AND lower(Q.Queue) NOT LIKE '%_current%'
										 AND lower(Q.Queue) NOT LIKE 'emp_tempassign%'
										 AND lower(Q.Queue) NOT LIKE 'vlt / negb%'
										 AND NOT ( lower(Q.Queue) LIKE '%ext' AND A.Channel IS NULL)
												 )							THEN 1
																			ELSE 2
							   END ASC,
							   Q.BusinessDate DESC)						AS RNK		
	, RANK() OVER (PARTITION BY Q.CaseCodeKey
		ORDER BY Q.BusinessDate DESC)							        AS RNKRefDate	
FROM QueuesMtD AS Q
LEFT JOIN AssignmentMtD AS A ON Q.CaseCodeKey = A.CaseCode AND A.BusinessDate = Q.BusinessDate
'''
)

AllQueuesChannels.createOrReplaceTempView("AllQueuesChannels")

##### QueuesChannels

In [0]:
QueuesChannels = spark.sql(
f'''
SELECT
					QQ.CaseCodeKey
				,	QQ.BusinessDate
				,	QQ.Queue
				,	QQ.Channel
				,	QQ.Team
   				,	QQ.AgentFullName
				,	QQ.AgentPartyKey
				,	CASE WHEN QMAP.CourseOfAction  LIKE '%3869%'							THEN QMAP.CourseOfAction
						 WHEN QQ.Channel IN ('DNC Collection', 'Outsourcing Collection') 	THEN 'Collection'
						 WHEN QQ.Channel IN ('L3869 ΔΤ - FAs', 'FAs', 'IRT') 				THEN 'Restructuring'
						 WHEN QQ.Channel = 'Unhandled'							 			THEN 'Unhandled'
																							ELSE NULL
					END AS CourseOfAction
				,	QMAP.AssignmentSegmentDetailed
		FROM (
				SELECT 
							QC.CaseCodeKey
						,	QC.BusinessDate
						,	CASE WHEN QC.Channel IS NULL THEN RnkRef.Queue ELSE QC.Queue END		AS Queue
						,	IFNULL(QC.Channel, 'Unhandled')											AS Channel
						,	QC.Team
						,	QC.AgentFullName
						,	QC.AgentPartyKey
				FROM AllQueuesChannels		AS QC
				LEFT JOIN AllQueuesChannels  AS RnkRef	ON QC.CaseCodeKey = RnkRef.CaseCodeKey AND RnkRef.RNKRefDateHistorical = 1 
				WHERE 1=1
					  AND QC.RNKHistorical = 1
					  AND QC.BusinessDate <= '{AssignmentCutOffDate}'
		) QQ
		LEFT JOIN QueuesMapping AS QMAP	 ON	 QQ.Queue = QMAP.QueueDescription	
'''
)

QueuesChannels.createOrReplaceTempView("QueuesChannels")

##### QueuesChannelsForAddIntraColl

In [0]:
QueuesChannelsForAddIntraColl = spark.sql(
f'''
SELECT
					QQ.CaseCodeKey
				,	QQ.BusinessDate
				,	QQ.Queue
				,	QQ.Channel
				,	QQ.Team
				,	QQ.AgentFullName
				,	QQ.AgentPartyKey
				,	CASE WHEN QMAP.CourseOfAction  LIKE '%3869%'							THEN QMAP.CourseOfAction
						 WHEN QQ.Channel IN ('DNC Collection', 'Outsourcing Collection') 	THEN 'Collection'
						 WHEN QQ.Channel IN ('L3869 ΔΤ - FAs', 'FAs', 'IRT') 				THEN 'Restructuring'
						 WHEN QQ.Channel = 'Unhandled'							 			THEN 'Unhandled'
																							ELSE NULL
					END AS CourseOfAction
				,	QMAP.AssignmentSegmentDetailed
		FROM (
				SELECT 
							QC.CaseCodeKey
						,	QC.BusinessDate
						,	CASE WHEN QC.Channel IS NULL THEN RnkRef.Queue ELSE QC.Queue END		AS Queue
						,	IFNULL(QC.Channel, 'Unhandled')											AS Channel
						,	QC.Team
						,	QC.AgentFullName
						,	QC.AgentPartyKey
				FROM AllQueuesChannels		AS QC
				LEFT JOIN AllQueuesChannels  AS RnkRef	ON QC.CaseCodeKey = RnkRef.CaseCodeKey AND RnkRef.RNKRefDate = 1 
				WHERE 1=1
					  AND QC.RNK = 1
		) QQ
		LEFT JOIN QueuesMapping AS QMAP	 ON	 QQ.Queue = QMAP.QueueDescription	
'''
)

QueuesChannelsForAddIntraColl.createOrReplaceTempView("QueuesChannelsForAddIntraColl")

##### SOMAccounts

In [0]:
SOMAccounts = spark.sql(
f'''
SELECT 
        '{run_date}' AS BusinessDate
	, E.PortfolioName
      , E.ExposureKey                   
      , E.AgreementKey                  
      , E.CDI                           
      , E.PartyTaxRegistrationNumber    
      , E.AssetSegment
      , E.ProductKey
      , AQ.OnBalance                                        AS SoMOnBalanceAmount
      , cast(AQ.OnBalance / coalesce( ER.ExchangeRate, 1)  as decimal(19,2))      AS SoMOnBalanceAmountEUR
      , E.GBV                                               AS SoMGBV
      , cast(E.GBV / coalesce( ER.ExchangeRate, 1)  as decimal(19,2))             AS SoMGBVEUR
      , AQ.Bucket                                           AS SoMBucket
      , E.DaysPastDueWithMateriality                        AS SoMDaysPastDue
      , E.ExposureStatus                                    AS SoMExposureStatus
      , AQ.ObjectiveStatusCode                              AS SoMObjectiveStatusCode
      , AQ.ObjectiveStatusdate                              AS SoMObjectiveStatusDate
      , CD.IsActiveFlag                                     AS MtDIsActiveFlag
      , CD.ClosedDate                                       AS MtDClosedDate    
	, QM.Channel                                          AS AssignmentChannelCalc
	, CASE WHEN QM.Channel <>'Outsourcing Collection' AND lower(QM.Team) LIKE 'pre%'    THEN 'Unassigned'
						                                                      ELSE QM.Team
	  END	AS AssignmentTeamCalc
	, QM.AgentFullName                                     AS AssignmentAgentCalc	
      , QM.AgentPartyKey	                               AS AssignmentAgentKeyCalc
      , QM.Queue                                             AS QueueDescriptionFinal
	, QM.CourseOfAction
      , QM.AssignmentSegmentDetailed
      , AQ.CaseCode      
FROM PerimeterProfitsSOM            AS E
LEFT JOIN ExchangeRatesSoM          AS ER ON E.Currency = ER.OriginalCurrency
LEFT JOIN AccountsQualcoSoM         AS AQ ON AQ.ExposureKey =  RIGHT(E.ExposureKey, LEN(E.ExposureKey) - 3) 
LEFT JOIN ClosedDateMtD             AS CD ON CD.ExposureKey = E.ExposureKey AND CD.Rank = 1
LEFT JOIN QueuesChannels            AS QM ON QM.CaseCodeKey = AQ.CaseCode
WHERE E.GBV > 0
AND E.ExposureStatus <> 'Writen Off'
'''
)

SOMAccounts.createOrReplaceTempView("SOMAccounts")

##### DateCollectionAssigmentv1

In [0]:
DateCollectionAssigmentv1 = spark.sql(
f'''
SELECT 
		A.CaseCodeKey
	,	MIN(A.BusinessDate) MinDate 
FROM AllQueuesChannels A
INNER JOIN SOMAccounts B ON A.CaseCodeKey =B.CaseCode AND B.AssignmentChannelCalc = 'Outsourcing Collection'
WHERE lower(A.Queue) LIKE '%ext' AND lower(A.Team) LIKE 'pre%'
AND A.BusinessDate <= '{AssignmentCutOffDate}'
GROUP BY a.CaseCodeKey
'''
)

DateCollectionAssigmentv1.createOrReplaceTempView("DateCollectionAssigmentv1")

##### DateCollectionAssigmentv1ForAddIntraColl

In [0]:
DateCollectionAssigmentv1ForAddIntraColl = spark.sql(
f'''
SELECT 
		A.CaseCodeKey
	,	MIN(A.BusinessDate) MinDate 
FROM AllQueuesChannels A
INNER JOIN QueuesChannelsForAddIntraColl B ON A.CaseCodeKey =B.CaseCodeKey AND B.Channel = 'Outsourcing Collection'
WHERE lower(A.Queue) LIKE '%ext' AND lower(A.Team) LIKE 'pre%'
GROUP BY a.CaseCodeKey
'''
)

DateCollectionAssigmentv1ForAddIntraColl.createOrReplaceTempView("DateCollectionAssigmentv1ForAddIntraColl")

##### DateCollectionAssigmentv2

In [0]:
DateCollectionAssigmentv2 = spark.sql(
f'''
SELECT 
		A.CaseCodeKey
	,	MIN(A.BusinessDate) MinDate
	,	MAX(A.BusinessDate) MaxDate
FROM AllQueuesChannels A
INNER JOIN SOMAccounts B ON A.CaseCodeKey =B.CaseCode AND B.AssignmentChannelCalc='Outsourcing Collection'
WHERE (
			 lower(IFNULL(A.Queue,'N/A')) 	NOT LIKE '%ext' AND lower(IFNULL(A.Team,'n/a')) NOT LIKE 'pre%' 
		 AND lower(IFNULL(A.Queue,'n/a'))	NOT LIKE '%low_debt%'
		 AND lower(IFNULL(A.Queue,'n/a'))	NOT LIKE '%under_11_dpds%'
		 AND lower(IFNULL(A.Queue,'n/a'))	NOT LIKE '%_current%'
		 AND lower(IFNULL(A.Queue,'n/a'))	NOT LIKE 'emp_tempassign%'
		 AND lower(IFNULL(A.Queue,'n/a'))	NOT LIKE 'vlt / negb%'
		)
AND A.BusinessDate <= '{AssignmentCutOffDate}' 
GROUP BY A.CaseCodeKey
'''
)

DateCollectionAssigmentv2.createOrReplaceTempView("DateCollectionAssigmentv2")

##### SOMAccountsv2ForAddIntraColl

In [0]:
SOMAccountsv2ForAddIntraColl = spark.sql(
f'''
SELECT 
		A.CaseCodeKey
	,	CASE WHEN a.Channel='Outsourcing Collection' AND B.MinDate> '{AssignmentCutOffDate}'	THEN 2
			                                                                                    ELSE 0 
        END AS CollectionIntramonthAssignment
FROM QueuesChannelsForAddIntraColl A
LEFT JOIN DateCollectionAssigmentv1ForAddIntraColl B ON A.CaseCodeKey=B.CaseCodeKey
WHERE a.Channel='Outsourcing Collection'
'''
)

SOMAccountsv2ForAddIntraColl.createOrReplaceTempView("SOMAccountsv2ForAddIntraColl")

##### SOMAccountsFinal

In [0]:
SOMAccountsFinal = spark.sql(
f'''
SELECT 
		A.*
	,	CASE WHEN A.AssignmentChannelCalc='Outsourcing Collection' AND B.MinDate>= '{FirstAssignmentDate}'	AND B.MINDATE<='{AssignmentCutOffDate}' THEN 1
			 WHEN A.AssignmentChannelCalc='Outsourcing Collection' AND B.MinDate>C.MaxDate AND B.MinDate>= '{FirstAssignmentDate}'	AND B.MINDATE <='{AssignmentCutOffDate}' THEN 1
			 WHEN A.AssignmentChannelCalc='Outsourcing Collection' AND B.MinDate< C.MinDate		AND B.MINDATE<='{AssignmentCutOffDate}' THEN 1
			 WHEN D.CollectionIntramonthAssignment = 2																													   										THEN 2
			 																																																													ELSE 0 
    END AS CollectionIntramonthAssignment
  , CASE WHEN A.AssignmentChannelCalc IS NOT NULL AND A.AssignmentChannelCalc <> 'Unhandled' 											THEN 'Assigned'
				 WHEN a.SoMBucket = 0		 																							THEN 'Bucket 0'
				 WHEN a.SoMDaysPastDue < 11 AND A.SoMExposureStatus <> 'Define Delay' AND A.SoMBucket < 13								THEN '1-10 dpd'
				 WHEN A.CourseOfAction LIKE '%3869%'										THEN 'N.3869 passive'
				 WHEN lower(A.AssignmentSegmentDetailed) LIKE '%low debt%' OR lower(A.AssignmentSegmentDetailed) LIKE '%low ticket%'	THEN 'Low Debt'		
				 -------- QUEUE MAPPING
				 WHEN a.QueueDescriptionFinal = 'Active_Settlements'			THEN 'Active DPO settlement'	
				 WHEN a.QueueDescriptionFinal = 'All_Borrowers_Deceased'		THEN 'All borrowers deceased'	
				 WHEN a.QueueDescriptionFinal = 'CASE_WATCH_LIST'				THEN 'CASE_WATCH_LIST'	
				 WHEN lower(a.QueueDescriptionFinal) LIKE 'collection_sapphire%'		THEN 'Collection_Sapphire'	
				 WHEN a.QueueDescriptionFinal = 'Cosmos_Foreign_Currency_c'		THEN 'Cosmos_Foreign_Currency_c'	
				 WHEN a.QueueDescriptionFinal = 'Ex_Wholesale_Judicial'			THEN 'Ex_Wholesale_Judicial'	
				 WHEN a.QueueDescriptionFinal = 'DENOUNCED-WOFF_GOVERMENT_GUARANTEE LOANS'		THEN 'Gov_Guar'	
				 WHEN a.QueueDescriptionFinal IN ('Cosmos_Low_Debt_c', 'Galaxy_Low_Debt_g',  'Securitized_Gefyra_Low_Debt', 'VLT / NegB_c', 'VLT / NegB_g')	THEN 'Low Debt'
				 WHEN a.QueueDescriptionFinal IN ('N.4605_ΟΡΙΣΤΙΚΕΣ_ΑΠΟΦΑΣΕΙΣ', 'N.4605_ΠΡΟΣΩΡΙΝΕΣ_ΔΙΑΤΑΓΕΣ', 
				 			'N3869_G_Modified_g2ext', 'N3869_G_Modified_gext', 'N3869_S_Modified_cext', 
				 			'N3869_Εξαίρεση_διαχείρισης_Παράπονα', 'Settlements_Προσωρινών_Διαταγών_3869',
				 			'Settlements_Προσωρινών_Διαταγών_3869_All', 'ΔΤ3869_2706_(GALAXY_TEAM_3)',
				 			'ΔΤ3869_NO_TEAM', 'ΔΤ3869_Παρελθοντική Δικάσιμος', 
				 			'ΔΤ3869_Παρελθοντική Δικάσιμος με ΗμερΟριστΔικασίμ', 'Ν.3869-N.4605_ΧΡΕΩΣΗ_ΕΞΟΔΩΝ',
				 			'Υλοποιημένη Δικ.Αποφ.Ν3869_Other Contract Case')																										THEN 'N.3869 passive'		
				 WHEN a.QueueDescriptionFinal = 'N.3869_Έναρξη_Διαδικασίας_Έκπτωσης'		THEN 'N.3869_Έναρξη_Διαδικασίας_Έκπτωσης'										
				 WHEN a.QueueDescriptionFinal IN ('New_OCW_Active_Application', 'OCW Εξωδικαστικός Συμβιβασμός Ν4469')	THEN 'OCW'
				 WHEN a.QueueDescriptionFinal IN ('Oristikes_Apofaseis_N3869_Current', 'Oristikes_Apofaseis_N3869_Den/Woff', 'Oristikes_Apofaseis_N3869_Low_Debt',
				 			'Oristikes_Apofaseis_N3869_Settlement')									THEN 'Oristikes_Apofaseis_N3869'
				 WHEN a.QueueDescriptionFinal IN ('LEGAL - ΠΡΟΠΤΩΧΕΥΤΙΚΗ ΔΙΑΔΙΚΑΣΙΑ', 'LEGAL - ΠΤΩΧΕΥΣΗ ΕΠΙΧΕΙΡΗΣΕΩΝ')	THEN 'Πτωχευμένες εταιρείες'
																														ELSE 'Rest' 
				 END AS AccountCategory
	, CASE WHEN SUBSTRING(A.ExposureKey,4,4) = '4000' THEN 'G_Modified' ELSE NULL END AS GModifiedAccount
	, SEG.BusinessDate AS SegmentationBusinessDate
	, SEG.OfficialSegment
	, SEG.NewOrAlphaProduct
	, SEG.BorrowerTotalGBV
	, SEG.BorrowerTotalGBVRange
	, SEG.HighBalanceFlag
	, PF.PayerFlagEntity
FROM SOMAccounts A
LEFT JOIN DateCollectionAssigmentv1	B ON A.CaseCode =B.CaseCodeKey
LEFT JOIN DateCollectionAssigmentv2	C ON A.CaseCode=C.CaseCodeKey
LEFT JOIN SOMAccountsv2ForAddIntraColl	D ON A.CaseCode=D.CaseCodeKey
LEFT JOIN SegmentationSoM	SEG ON SEG.ExposureKey=  RIGHT(A.ExposureKey, LEN(A.ExposureKey) - 3) 
LEFT JOIN PayersFlagAlgo PF ON PF.CDI = A.CDI AND PF.PortfolioName = A.PortfolioName
'''
)

SOMAccountsFinal.createOrReplaceTempView("SOMAccountsFinal")

##### final_df

In [0]:
final_df = spark.sql(
f'''
SELECT 
  CAST(A.BusinessDate AS DATE) AS BusinessDate
, CAST(A.PortfolioName AS STRING) AS PortfolioName
, CAST(A.ExposureKey AS STRING) AS ExposureKey
, CAST(A.AgreementKey AS STRING) AS AgreementKey
, CAST(A.CDI AS STRING) AS CDI
, CAST(A.PartyTaxRegistrationNumber AS STRING) AS PartyTaxRegistrationNumber
, CAST(A.AssetSegment AS STRING) AS AssetSegment
, CAST(A.ProductKey AS STRING) AS ProductKey
, CAST(A.SoMOnBalanceAmount AS DECIMAL(19,2)) AS SoMOnBalanceAmount
, CAST(A.SoMOnBalanceAmountEUR AS DECIMAL(19,2)) AS SoMOnBalanceAmountEUR
, CAST(A.SoMGBV AS DECIMAL(19,2)) AS SoMGBV
, CAST(A.SoMGBVEUR AS DECIMAL(19,2)) AS SoMGBVEUR
, CAST(A.SoMBucket AS STRING) AS SoMBucket
, CAST(A.SoMDaysPastDue AS INT) AS SoMDaysPastDue
, CAST(A.SoMExposureStatus AS STRING) AS SoMExposureStatus
--, CAST(A.SoMObjectiveStatus AS STRING) AS SoMObjectiveStatus
, CAST(A.SoMObjectiveStatusCode AS STRING) AS SoMObjectiveStatusCode
, CAST(A.SoMObjectiveStatusDate AS DATE) AS SoMObjectiveStatusDate
, CAST(A.MtDIsActiveFlag AS STRING) AS MtDIsActiveFlag
, CAST(A.MtDClosedDate AS DATE) AS MtDCloseDate
, CAST(A.AssignmentChannelCalc AS STRING) AS AssignmentChannel
, CAST(A.AssignmentTeamCalc AS STRING) AS AssignmentTeam
, CAST(A.AssignmentAgentCalc AS STRING) AS AssignmentAgent
, CAST(A.AssignmentAgentKeyCalc AS STRING) AS AssignmentAgentKey
, CAST(A.QueueDescriptionFinal AS STRING) AS QueueDescriptionFinal
, CAST(A.CourseOfAction AS STRING) AS CourseOfAction
, CAST(A.AssignmentSegmentDetailed AS STRING) AS AssignmentSegmentDetailed
, CAST(A.CaseCode AS STRING) AS CaseCode
, CAST(A.CollectionIntramonthAssignment AS INT) AS CollectionIntramonthAssignment
, CAST(A.AccountCategory AS STRING) AS AccountCategory
, CAST(A.GModifiedAccount AS STRING) AS GModifiedAccount
, CAST(A.SegmentationBusinessDate AS DATE) AS SegmentationBusinessDate
, CAST(A.OfficialSegment AS STRING) AS OfficialSegment
, CAST(A.NewOrAlphaProduct AS STRING) AS NewOrAlphaProduct
, CAST(A.PayerFlagEntity AS STRING) AS PayerFlagEntity    
, CAST(A.BorrowerTotalGBV AS DECIMAL(19,2)) AS BorrowerTotalGBV
, CAST(A.BorrowerTotalGBVRange AS STRING) AS BorrowerTotalGBVRange
, CAST(A.HighBalanceFlag AS INT) AS HighBalanceFlag
, CAST(getdate() AS TIMESTAMP) AS InsertTimeStamp
FROM SOMAccountsFinal A
'''
)
final_df.createOrReplaceTempView("final_df")

In [0]:
final_df.write \
    .format("delta") \
    .partitionBy("BusinessDate") \
    .mode("append") \
    .saveAsTable(delivery_table_path) # in order to write in delta_lake

In [0]:
%python
from pyspark.sql.functions import col, count, concat

def check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error"):
    errors = []
    warnings = []

    if key_columns:
        # Create the composite key column if there are multiple key columns
        if len(key_columns) > 1:
            composite_key = concat(*[col(c) for c in key_columns]).alias("CompositeKey")
            key_column_name = "CompositeKey"
            key_columns_string = ", ".join(key_columns)
        else:
            key_column_name = key_columns[0]
            key_columns_string = key_columns[0]

        # Duplicate Checks
        duplicate_count = (
            spark.table(delivery_table_path)
            .filter(col("BusinessDate") == run_date)
            .withColumn("CompositeKey", composite_key if len(key_columns) > 1 else col(key_column_name))
            .groupBy(key_column_name)
            .count()
            .filter("count > 1")
            .select(key_column_name, "count")
            .toPandas()
            .to_dict(orient='records')
        )

        if duplicate_count:
            message = f"Duplicate {key_columns_string} entries found: {duplicate_count}"
            if duplicate_check_action == "error":
                errors.append(message)
            elif duplicate_check_action == "warning":
                warnings.append(message)

    # Check if the table for the specific date is empty
    row_count = spark.sql(f"""
        SELECT * FROM {delivery_table_path}
        WHERE BusinessDate = '{run_date}'
    """).count()

    if row_count == 0:
        message = f"The {delivery_table_path} dataset is empty for the date {run_date}."
        if empty_check_action == "error":
            errors.append(message)
        elif empty_check_action == "warning":
            warnings.append(message)

    if warnings:
        print("Warnings:\n" + "\n".join(warnings))

    if errors:
        raise ValueError("Errors found:\n" + "\n".join(errors))

key_columns = ["ExposureKey"]
check_table(delivery_table_path, run_date, key_columns, duplicate_check_action="error", empty_check_action="error")

In [0]:
import re
def rename_and_cleanup_files(output_path, file_name):
    part_file_regex = re.compile(r'part-.*')
    system_file_regex = re.compile(r'^_(committed_|started_|SUCCESS)')
 
    # Move the part file to the final filename.
    files = dbutils.fs.ls(output_path)
    for file in files:
        print(f"file: {file.name}")
        if part_file_regex.match(file.name):
            original_path = file.path
            new_path = output_path + file_name
            dbutils.fs.mv(original_path, new_path)
 
    # Remove system files from the common folder.
    files = dbutils.fs.ls(output_path)
    for file in files:
        if system_file_regex.match(file.name):
            dbutils.fs.rm(file.path, recurse=False)
 
# Example usage:
# rename_and_cleanup_files(output_path, file_name)

In [0]:
Assignment = spark.sql(
f'''
select * from {delivery_table_path}
where BusinessDate = {run_date!r}
'''
)
Assignment.createOrReplaceTempView("Assignment")

In [0]:
output_path = "/Volumes/"+env_prefix+"-cdh-delivery/repoint/exports/HAPS/AssignmentMapping/" # 
file_name = "AssignmentMapping_" +run_date+ ".csv"
Assignment.coalesce(1).write.mode("append").option("header", True).csv(f"{output_path}") # 
rename_and_cleanup_files(output_path, file_name)